hello


In [2]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_community.llms import Ollama

from sql.sqlDB import sql_tools

# =====================================================
# STATE
# =====================================================
class State(TypedDict):
    messages: List


# =====================================================
# TOOLS (MCP SIMULATION)
# =====================================================

@tool
def erp_tool(query: str):
    """Fetch ERP data"""
    return {"vendor_cost": 300000}


@tool
def rag_tool(query: str):
    """Fetch company documents"""
    return {"docs": ["Q2 report", "CEO notes"]}


tools = [sql_tools, erp_tool, rag_tool]


# =====================================================
# LLM WITH TOOLS BOUND
# =====================================================
llm = Ollama(
    model="llama3",    #later change to open ai
    
)
llm_with_tools = llm.bind_tools(tools)


# =====================================================
# AGENT NODE (LLM)
# =====================================================
def agent(state: State):
    messages = state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": messages + [response]}


# =====================================================
# TOOL NODE (EXECUTES TOOL CALLS)
# =====================================================
tool_node = ToolNode(tools)


# =====================================================
# GRAPH BUILD
# =====================================================
graph = StateGraph(State)

graph.add_node("agent", agent)
graph.add_node("tools", tool_node)

# ENTRY POINT
graph.set_entry_point("agent")


# =====================================================
# CRITICAL PART: CONDITIONAL EDGE
# =====================================================

graph.add_conditional_edges(
    "agent",
    tools_condition,   # 👈 THIS decides: tool OR end
    {
        "tools": "tools",
        END: END
    }
)

# after tool execution → go back to agent
graph.add_edge("tools", "agent")


# =====================================================
# COMPILE
# =====================================================
app = graph.compile()


# =====================================================
# RUN
# =====================================================
if __name__ == "__main__":
    result = app.invoke({
        "messages": [
            {
                "role": "user",
                "content": "Why did profit decrease in Q2?"
            }
        ]
    })

    print(result["messages"][-1].content)

ModuleNotFoundError: No module named 'langchain_community'